# Module 01 · Kuhn Poker

Train CFR on Kuhn Poker, inspect the strategy, and measure exploitability. Run from the repo root after `pip install -e ".[dev]"`.

In [ ]:
from pokerbot.games.kuhn import KuhnPoker
from pokerbot.solvers.cfr import CFRTrainer
from pokerbot.eval.exploitability import exploitability, expected_value, best_response_value
import matplotlib.pyplot as plt

game = KuhnPoker()

## 1. Train and read the strategy
Each entry is `[P(pass), P(bet)]`. Keys are *card + history*, i.e. one information set each.

In [ ]:
trainer = CFRTrainer(game)
trainer.train(20000)
strategy = trainer.average_strategy()
for k in sorted(strategy):
    print(f"{k:>4}  pass={strategy[k][0]:.3f}  bet={strategy[k][1]:.3f}")
print("game value for player 0:", round(expected_value(game, strategy, 0), 4), " (Nash: -1/18 = -0.0556)")

## 2. Exploitability vs iterations
The scoreboard for the whole repo. Expect a roughly power-law decline.

In [ ]:
trainer = CFRTrainer(game)
xs, ys = [], []
def record(t):
    xs.append(t.iterations)
    ys.append(exploitability(game, t.average_strategy()))
trainer.train(5000, callback=record, callback_every=100)
plt.loglog(xs, ys); plt.xlabel("iterations"); plt.ylabel("exploitability"); plt.grid(True, which="both", alpha=.3)
plt.title("Vanilla CFR on Kuhn Poker"); plt.show()

## 3. Check your own equilibrium
Pick `alpha` in [0, 1/3], build the whole strategy by hand (see the table in the docs page), and verify exploitability is 0.

In [ ]:
alpha = 1/6
mine = {
    # player 0 opening
    "J": [1-alpha, alpha], "Q": [1.0, 0.0], "K": [1-3*alpha, 3*alpha],
    # player 0 after check-bet
    "Jpb": [1.0, 0.0], "Qpb": [1-(alpha+1/3), alpha+1/3], "Kpb": [0.0, 1.0],
    # player 1 after a check
    "Jp": [2/3, 1/3], "Qp": [1.0, 0.0], "Kp": [0.0, 1.0],
    # player 1 facing a bet
    "Jb": [1.0, 0.0], "Qb": [2/3, 1/3], "Kb": [0.0, 1.0],
}
print("exploitability:", exploitability(game, mine))
print("value for player 0:", expected_value(game, mine, 0))

## 4. Break it
Nudge one probability and see how much a best-responding opponent gains. Which information set is the most sensitive?

In [ ]:
import copy
for key in ["Qpb", "Jp", "Qb"]:
    broken = copy.deepcopy(mine)
    p = broken[key][1]
    broken[key] = [1-(p+0.1), p+0.1] if p + 0.1 <= 1 else [0.0, 1.0]
    print(f"{key}: exploitability after +0.1 bet prob = {exploitability(game, broken):.4f}")